In [6]:
import time
from dotenv import load_dotenv
load_dotenv()
from app.services.db_service import DbService
sql_db_service = DbService()
await sql_db_service.init_pool()

from app.services.external_api.searoute_api import SearouteApi
searoute_api = SearouteApi("https://api.searoutes.com/", "EWhTo2x2hihNDCPZjCaMgFDWGegJoVLnYP7mqi5L")

import asyncpg
from app.data.dto.main.SeaPort import SeaPortDB, SeaPort
db_name="BunkeringBot"
db_user="def_user"
db_password="super_password"
db_host="77.37.96.222"
db_port="5432"


connection_pool = await asyncpg.create_pool(
            database=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
            min_size=1,
            max_size=20
        )

In [6]:
ports = []

async with connection_pool.acquire() as conn:
    rows = await conn.fetch(
        """
        SELECT *
        FROM public.ports_vector_new
        WHERE port_size is Null
        """
    )

    for r in rows:
        p = SeaPortDB.from_db_row(r)
        ports.append(p)

len(ports)

4397

In [9]:
ports[0]


In [12]:
found = []
not_found = []

for port in ports[0:1000]:
    s_port, err = await searoute_api.get_port_info(port.locode.strip())
    if s_port and not err:
        found.append(s_port)
    else:
        not_found.append(port)

    time.sleep(0.7)


In [8]:
from app.data.dto.searoute.SearoutePort import SearoutePort

In [16]:
import json

with open("seaports.json", 'w') as fp:
    json.dump([f.model_dump() for f in found], fp)

In [13]:
import json
with open("seaports.json", 'r') as fp:
    ports_raw = json.load(fp)
    ports = [SearoutePort.from_dict(r) for r in ports_raw]

In [21]:
errored_ports = [p for p in ports if p.locode in set(errors)]

In [29]:
errored_ports

[SearoutePort(name='Issigeac', locode='FRIGC', country='France', countryCode='FR', countryName='France', size='tiny', eta_datetime=None, distance=None, latitude=44.733333333333334, longitude=0.6),
 SearoutePort(name='Trana', locode='NOTRA', country='Norway', countryCode='NO', countryName='Norway', size='tiny', eta_datetime=None, distance=None, latitude=66.498, longitude=12.0935),
 SearoutePort(name='Corconne', locode='FRZNN', country='France', countryCode='FR', countryName='France', size='tiny', eta_datetime=None, distance=None, latitude=43.87231, longitude=3.93959),
 SearoutePort(name='Espevik', locode='NOESP', country='Norway', countryCode='NO', countryName='Norway', size='tiny', eta_datetime=None, distance=None, latitude=59.31666666666667, longitude=5.666666666666667),
 SearoutePort(name='Betbezer', locode='FR4FQ', country='France', countryCode='FR', countryName='France', size='tiny', eta_datetime=None, distance=None, latitude=43.96666666666667, longitude=-10.0)]

In [30]:
u_s = []
u_err = []

for p in errored_ports:
    p, err = await sql_db_service.upsert_port_size_from_searoute(p)
    if p and not err:
        u_s.append(p)
    else:
        u_err.append((p, err))

In [31]:
len(u_s)

5

In [23]:
s = []
errors = []

for p in errored_ports:
    np, err = await sql_db_service.update_port(p.locode, {'port_size': p.size})
    if np and not err:
        s.append(p.locode)
    else:
        errors.append((p.locode, err))

In [24]:
errors

[('FRIGC', 'Port not found'),
 ('NOTRA', 'Port not found'),
 ('FRZNN', 'Port not found'),
 ('NOESP', 'Port not found'),
 ('FR4FQ', 'Port not found')]

In [28]:
p, err = await sql_db_service.get_port_by_locode(errors[0][0])
err

'Could not find port'